In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("london_energy.csv")

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort data
df = df.sort_values(['LCLid', 'Date'])

# Reduce dataset (50 households)
ids = df['LCLid'].unique()[:50]
df = df[df['LCLid'].isin(ids)]

print(df.shape)

(36828, 3)


In [ ]:
# Time features
df['day'] = df['Date'].dt.day
df['month'] = df['Date'].dt.month
df['weekday'] = df['Date'].dt.weekday

# Lag features
df['lag_1'] = df.groupby('LCLid')['KWH'].shift(1)
df['lag_2'] = df.groupby('LCLid')['KWH'].shift(2)
df['lag_3'] = df.groupby('LCLid')['KWH'].shift(3)

# Rolling mean (last 3 days)
df['rolling_mean_3'] = df.groupby('LCLid')['KWH'].shift(1).rolling(3).mean()

# Drop missing values
df = df.dropna()
print(df.head())

        LCLid       Date     KWH  day  month  weekday   lag_1   lag_2   lag_3  \
6   MAC000002 2012-10-18  10.751   18     10        3  10.885   9.769  10.257   
7   MAC000002 2012-10-19   8.431   19     10        4  10.751  10.885   9.769   
8   MAC000002 2012-10-20  17.578   20     10        5   8.431  10.751  10.885   
9   MAC000002 2012-10-21  24.490   21     10        6  17.578   8.431  10.751   
10  MAC000002 2012-10-22  18.885   22     10        0  24.490  17.578   8.431   

    rolling_mean_3  
6        10.303667  
7        10.468333  
8        10.022333  
9        12.253333  
10       16.833000  


In [ ]:
X = df[['day', 'month', 'weekday', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']]
y = df['KWH']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Models
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100)
xgb = XGBRegressor()

# Train
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name, model in models.items():
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    print(name)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)
    print("------")

Linear Regression
MAE: 2.8451832075433217
RMSE: 4.427361549945551
R2: 0.8450029326911064
------
Random Forest
MAE: 2.9555006487818227
RMSE: 4.606965003394172
R2: 0.8321724211860233
------
XGBoost
MAE: 2.976276278098452
RMSE: 4.695859508425212
R2: 0.8256332418572894
------


In [ ]:
import joblib

joblib.dump(lr, "energy_model.pkl")

['energy_model.pkl']

In [ ]:
from google.colab import files
files.download("energy_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>